# 🏡 Machine Learning Pipeline: Prediksi Harga Rumah (Surabaya)

Notebook ini adalah pipeline Data Science LENGKAP untuk membangun model AI yang memprediksi **harga rumah di Surabaya**.

### 🎯 Tujuan:
Membangun model Machine Learning yang bisa memprediksi **harga rumah** berdasarkan spesifikasi fisiknya.

### 🛠️ Algoritma yang Digunakan:
1. **Linear Regression** — Model statistik klasik
2. **Random Forest** — Ensemble learning berbasis Decision Tree
3. **Neural Network** — Deep Learning dengan Keras/TensorFlow

## 📦 Tahap 1: Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, joblib, warnings, shutil
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from google.colab import files

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

## 📂 Tahap 2: Upload Dataset\nUpload file **Dataset rumah123 CLEAN.csv**

In [ ]:
print("Silakan upload: Dataset rumah123 CLEAN.csv")
uploaded = files.upload()
file_key = list(uploaded.keys())[0]
df_raw = pd.read_csv(file_key) if file_key.endswith('.csv') else pd.read_excel(file_key)
print("✅ Berhasil dimuat! Ukuran:", df_raw.shape)
display(df_raw.head())

## 🧹 Tahap 3: Data Cleaning\nMenghapus kolom identitas (nomor urut, link URL, gambar, ID iklan) dan memperbaiki format data.

In [ ]:
df = df_raw.copy()

# Hapus kolom identitas (bukan spesifikasi rumah)
df = df.drop(columns=[c for c in ['index', 'Link', 'IMG Link', 'ID Iklan'] if c in df.columns])

# Isi missing values
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].replace(['-', '', ' '], 'Tidak Diketahui').fillna('Tidak Diketahui')
    else:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

print("Kolom setelah cleaning:", df.columns.tolist())
display(df.head())

## 📊 Tahap 4: Exploratory Data Analysis (EDA)

In [ ]:
# Distribusi Harga
plt.figure(figsize=(10, 5))
sns.histplot(df['Harga'], bins=50, kde=True, color='blue')
plt.title('Distribusi Harga Rumah Surabaya', fontsize=14, fontweight='bold')
plt.xlabel('Harga (Rupiah)')
plt.ylabel('Frekuensi')
plt.show()

# Heatmap Korelasi
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if len(num_cols) > 1:
    plt.figure(figsize=(10, 8))
    corr = df[num_cols].corr()
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
    plt.title('Heatmap Korelasi', fontsize=14, fontweight='bold')
    plt.show()

## 🎛️ Tahap 5: Feature Selection (Seleksi Fitur)

**Analisis Fitur Surabaya (21 kolom asli):**

| Kolom | Keputusan | Alasan |
|-------|-----------|--------|
| index | ❌ BUANG (di cleaning) | Nomor urut, tidak ada hubungan |
| Link | ❌ BUANG (di cleaning) | URL iklan, unik tiap baris (11496 unik) |
| IMG Link | ❌ BUANG (di cleaning) | URL gambar, unik tiap baris |
| ID Iklan | ❌ BUANG (di cleaning) | Kode iklan unik, tidak relevan |
| Alamat | ❌ BUANG | Hanya 7 nilai unik (kelurahan). Terlalu sedikit variasi untuk jadi fitur berguna |
| Tipe Properti | ❌ BUANG | Hanya 1 nilai ("Rumah"). Tidak ada variasi sama sekali |
| Dilengkapi Perabotan | ❌ BUANG | 8745 dari 11497 baris berisi "-" (76% kosong). Data terlalu sedikit |
| Hadap | ❌ BUANG | 9154 dari 11497 baris berisi "-" (80% kosong) |
| Carport | ❌ BUANG | 10424 dari 11497 baris berisi "-" (91% kosong) |
| Umur Bangunan | ❌ BUANG | 5788 dari 11497 baris berisi "-" (50% kosong) |
| Kamar Pembantu | ❌ BUANG | 7048 dari 11497 baris berisi "-" (61% kosong) |
| Kondisi Properti | ❌ BUANG | 7582 dari 11497 baris berisi "-" (66% kosong) |
| Luas Tanah | ✅ PAKAI | Data lengkap, korelasi kuat dengan harga |
| Luas Bangunan | ✅ PAKAI | Data lengkap, korelasi kuat |
| Kamar | ✅ PAKAI | Data lengkap, jumlah kamar penentu harga |
| Kamar Mandi | ✅ PAKAI | Data lengkap |
| Sertifikat | ✅ PAKAI | 508 kosong (4%), masih layak. SHM vs HGB mempengaruhi harga |
| Jumlah Lantai | ✅ PAKAI | Data lengkap, rumah bertingkat lebih mahal |
| Daya Listrik | ✅ PAKAI | 3304 kosong (29%), tapi daya listrik indikator kelas rumah |
| Garasi | ✅ PAKAI | Data lengkap, garasi menambah nilai rumah |

In [ ]:
# Kolom yang dibuang karena tidak berguna / noise
kolom_dibuang = ['Alamat', 'Tipe Properti', 'Dilengkapi Perabotan', 'Hadap', 'Carport', 'Umur Bangunan', 'Kamar Pembantu', 'Kondisi Properti']

df_selected = df.drop(columns=[c for c in kolom_dibuang if c in df.columns])

print("❌ DIBUANG (tidak berguna untuk prediksi):")
for c in kolom_dibuang:
    if c in df.columns:
        print(f"   ❌ {c}")

print("\n✅ DIPERTAHANKAN (berguna untuk prediksi):")
for c in df_selected.columns:
    if c != 'Harga':
        print(f"   ✅ {c}")

## ✂️ Tahap 6: Hapus Outlier (IQR) & Label Encoding

In [ ]:
# Hapus Outlier Harga
Q1 = df_selected['Harga'].quantile(0.25)
Q3 = df_selected['Harga'].quantile(0.75)
IQR = Q3 - Q1
df_clean = df_selected[(df_selected['Harga'] >= Q1 - 1.5*IQR) & (df_selected['Harga'] <= Q3 + 1.5*IQR)]
print(f"Data Awal: {len(df_selected)} | Setelah Outlier Dihapus: {len(df_clean)}")

# Label Encoding
df_final = df_clean.copy()
cat_cols = df_final.select_dtypes(include=['object']).columns.tolist()
features = [c for c in df_final.columns if c != 'Harga']
encoders = {}
schema = []

for col in features:
    if col in cat_cols:
        le = LabelEncoder()
        df_final[col] = le.fit_transform(df_final[col].astype(str))
        encoders[col] = le
        schema.append({'name': col, 'label': col, 'type': 'categorical', 'options': list(le.classes_)})
    else:
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce').fillna(0)
        schema.append({'name': col, 'label': col, 'type': 'numeric', 'min': float(df_final[col].min()), 'max': float(df_final[col].max()), 'default': float(df_final[col].median())})


## 🧠 Tahap 7: Training Model\nMelatih 3 algoritma dan membandingkan skor R2 (semakin dekat 1.0 = semakin bagus).

In [ ]:
X = df_final[features]
y = df_final['Harga']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
pred_rf = rf.predict(X_test_scaled)
r2_rf = r2_score(y_test, pred_rf)

# Linear Regression
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
pred_lr = lr.predict(X_test_scaled)
r2_lr = r2_score(y_test, pred_lr)

# Neural Network
model_nn = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2), Dense(64, activation='relu'), Dropout(0.2),
    Dense(32, activation='relu'), Dense(1, activation='sigmoid')
])
model_nn.compile(optimizer='adam', loss='mae', metrics=['mse'])
y_train_max = y_train.max()
model_nn.fit(X_train_scaled, y_train/y_train_max, validation_data=(X_test_scaled, y_test/y_train_max), epochs=100, batch_size=32, verbose=0)
pred_nn = model_nn.predict(X_test_scaled).flatten() * y_train_max
r2_nn = r2_score(y_test, pred_nn)

print(f"\n🏆 HASIL: RF={r2_rf:.3f} | LR={r2_lr:.3f} | NN={r2_nn:.3f}")

## 🎯 Tahap 8: Evaluasi & Feature Importance

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_test, pred_rf, alpha=0.5, color='purple')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
plt.title('Prediksi vs Aktual (Random Forest)', fontweight='bold')
plt.xlabel('Harga Asli'); plt.ylabel('Harga Prediksi')
plt.show()

fi = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)
plt.figure(figsize=(10, 6))
fi.plot(kind='barh', color='teal')
plt.title('Feature Importance (Fitur Paling Berpengaruh)', fontweight='bold')
plt.xlabel('Kepentingan')
plt.show()

## 📦 Tahap 9: Export Model

In [ ]:
folder = 'surabaya_models'
os.makedirs(folder, exist_ok=True)
joblib.dump(rf, f'{folder}/model_rf.pkl')
joblib.dump(lr, f'{folder}/model_lr.pkl')
joblib.dump(scaler, f'{folder}/scaler.pkl')
joblib.dump(encoders, f'{folder}/encoders.pkl')
model_nn.save(f'{folder}/model_nn.h5')

best_r2 = max(r2_rf, r2_lr, r2_nn)
best_name = 'Random Forest' if best_r2==r2_rf else ('Linear Regression' if best_r2==r2_lr else 'Neural Network')

metadata = {
    'train_size': len(X_train), 'test_size': len(X_test),
    'outliers_removed': len(df_selected)-len(df_clean),
    'clean_data': len(df_clean), 'total_data': len(df_selected),
    'features': features, 'schema': schema,
    'best_model': best_name, 'best_r2': float(best_r2),
    'y_max_nn': float(y_train_max),
    'feature_importances': dict(zip(features, map(float, rf.feature_importances_))),
    'lr_coef': dict(zip(features, map(float, lr.coef_))),
    'lr_intercept': float(lr.intercept_)
}
joblib.dump(metadata, f'{folder}/metadata.pkl')
shutil.make_archive(folder, 'zip', folder)
files.download(f'{folder}.zip')
print('🎉 Download berhasil!')